In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window


In [0]:
init_load_flag = int(dbutils.widgets.get("init_load_flag"))

In [0]:
%sql
CREATE OR REPLACE TABLE databricks_cat.silver.customers(
    customer_id INT,
    first_name STRING,
    last_name STRING,
    city STRING,
    email STRING
   
);

In [0]:
%sql
INSERT INTO databricks_cat.silver.customers VALUES
(1001, 'John', 'Doe', 'Chicago', 'john.doe@email.com'),
(1002, 'Jane', 'Smith', 'Houston', 'jane.smith@email.com'),
(1003, 'Mike', 'Brown', 'Dallas', 'mike.brown@email.com'),
(1004, 'Sarah', 'Wilson', 'Austin', 'sarah.wilson@email.com'),
(1005, 'David', 'Taylor', 'Phoenix', 'david.taylor@email.com'),
(1006, 'Emily', 'Johnson', 'Atlanta', 'emily.johnson@email.com'),
(1007, 'Chris', 'Miller', 'Denver', 'chris.miller@email.com'),
(1008, 'Olivia', 'Davis', 'Seattle', 'olivia.davis@email.com'),
(1009, 'Daniel', 'Moore', 'Miami', 'daniel.moore@email.com'),
(1010, 'Sophia', 'Anderson', 'Boston', 'sophia.anderson@email.com');

In [0]:
if init_load_flag == 0 :
    df_existing = spark.sql("SELECT * FROM databricks_cat.gold.Dmcustomers")
else:
    df = df.withColumn("dim_customer_key", monotonically_increasing_id()+lit(1)) \
            .withColumn("start_date",current_timestamp()) \
            .withColumn("end_date",lit("2999-12-31").cast("timestamp")) \
           .withColumn("is_current", lit("Y"))
    df= df.withColumn("dim_customer_key",col('dim_customer_key').cast("int"))


    df.write.mode("overwrite") \
            .format("delta") \
            .option("path","abfss://gold@datalakeimes.dfs.core.windows.net/DmCustomers") \
            .saveAsTable("databricks_cat.gold.DmCustomers")

    

    

In [0]:
df_existing.display()


### Add New Data

In [0]:
new_data = [
    (1003, 'Mike', 'Brown', 'San Francisco', 'mike.brown@email.com'),  # city changed
    (1005, 'David', 'Taylor', 'Las Vegas', 'david.taylor@email.com'),  # city changed
    (1011, 'New', 'User', 'Houston', 'new.user@email.com'), # new record
    (1012, 'Carter', 'Efe', 'Lagos', 'efe.c@gmail.com')     # new record
]

columns = ['customer_id', 'first_name', 'last_name', 'city', 'email']

incoming_df = spark.createDataFrame(new_data, columns)

display(incoming_df)

In [0]:
df_join = incoming_df.alias('new').join(df_existing.alias("dim"), incoming_df.customer_id == df_existing.customer_id, 'left')
display(df_join)

### Get Changed records

In [0]:
changed_df = df_join.filter(col('dim.customer_id').isNotNull()
                            & (col('new.city') != col('dim.city'))|
                            (col('new.email') != col('dim.email'))|
                            (col('new.first_name') != col('dim.first_name'))|
                            (col('new.last_name') != col('dim.last_name'))).select("new.*","dim.dim_customer_key")
                                                            





display(changed_df)


### Expire rows that have changes in existing df

In [0]:
from delta.tables import DeltaTable

In [0]:
dlt_obj = DeltaTable.forPath(spark,"abfss://gold@datalakeimes.dfs.core.windows.net/DmCustomers")
dlt_obj.alias("target").merge(changed_df.alias("source"), 
                              "target.customer_id = source.customer_id AND target.is_current = 'Y' ") \
        .whenMatchedUpdate(
            set = {
                "is_current":"'N'",
                "end_date":"current_timestamp()"
            }
        ).execute()

## Insert Update Customer Information to Gold Layer

In [0]:
update_df = changed_df.withColumn("customer_id", col("customer_id").cast("int")) \
                    .withColumn("start_date", current_timestamp()) \
                    .withColumn("end_date",lit("2999-12-31").cast("timestamp")) \
                    .withColumn("is_current", lit('Y'))

display(update_df)

update_df.write.mode("append") \
            .format("delta") \
            .option("path","abfss://gold@datalakeimes.dfs.core.windows.net/DmCustomers") \
            .saveAsTable("databricks_cat.gold.DmCustomers")


In [0]:
%sql
SELECT * FROM databricks_cat.gold.dmcustomers

In [0]:
# df_expired = df_existing.alias("dim").join(
#     changed_df.select('customer_id').alias('chg'),
#     'customer_id'
# ).withColumn(
#     'is_current', when(col('chg.customer_id').isNotNull(), lit('N')).otherwise(col('is_current'))
# ).withColumn(
#     'end_date', 
#     when(col('chg.customer_id').isNotNull(), current_timestamp())
#     .otherwise(col('end_date'))
# )

# display(df_expired)



### Brand New Data

In [0]:

max_dim_key = spark.sql("SELECT MAX(dim_customer_key) as max_key FROM databricks_cat.gold.dmcustomers")

max_key = max_dim_key.collect()[0]['max_key']

print(max_key)

In [0]:
df_new = df_join.filter(col('is_current').isNull()).select("new.*", "dim.dim_customer_key")



display(df_new)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, lit

window_spec = Window.orderBy("customer_id")

df_final = df_new.withColumn(
    "dim_customer_key",
    row_number().over(window_spec) + lit(max_key)
)

In [0]:


df_result = df_final.withColumn("customer_id", col("customer_id").cast("int")) \
                    .withColumn("start_date",current_timestamp()) \
                    .withColumn("end_date",lit("2999-12-31").cast("timestamp")) \
                    .withColumn("is_current", lit('Y'))

df_result.display()
                                
                            

In [0]:




df_result.write.mode("append") \
            .format("delta") \
            .option("path","abfss://gold@datalakeimes.dfs.core.windows.net/DmCustomers") \
            .saveAsTable("databricks_cat.gold.DmCustomers")

In [0]:
%sql
SELECT * FROM databricks_cat.gold.dmcustomers

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_cat.gold.employees
(
    id INT,
    name STRING,
    salary INT
)

-- INSERT INTO databricks_cat.gold.employees
-- VALUES (1, 'John', 50000),
--        (2, 'Jane', 60000),
--        (3, 'Bob', 70000),
--        (4, 'Alice', 80000)
    
-- SELECT * FROM databricks_cat.gold.employees
    


In [0]:
%sql
INSERT INTO databricks_cat.gold.employees
VALUES (1, 'John', 50000),
       (2, 'Jane', 60000),
       (3, 'Bob', 70000),
       (4, 'Alice', 80000)

In [0]:
%sql
SELECT * FROM databricks_cat.gold.employees

In [0]:
%sql
CREATE FUNCTION databricks_cat.gold.masking(salary STRING)
    RETURN CASE WHEN is_account_group_member('admins') THEN salary ELSE '**-**-**' END

In [0]:
%sql
ALTER TABLE databricks_cat.gold.employees
ALTER COLUMN salary SET MASK databricks_cat.gold.masking

In [0]:
%sql
SELECT * FROM databricks_cat.gold.employees

In [0]:
%sql
DROP FUNCTION databricks_cat.gold.masking

In [0]:
%sql
DROP TABLE databricks_cat.gold.employees